# ETL Pipeline
## Introduction
ETL stands for Extract, Transform and Load. As it's name we are extracting every token of datat from the rubble, clean them to transform into redable/useful data before being loaded into streams for future analysis/processing.

Ther are 3 layers of processing
- **Bronze Layer**: In this stage we extract meta-data such as file name, year, month, number of sheet and select desired sheet(s)
- **Silver Layer**: In this layer we deal with columns and data
- **Gold Layer**: Final transformation happens in this stage

<h3>Extracting Libraries</h3>

In [21]:
import os
import pandas as pd
from openpyxl import load_workbook
from pyxlsb import open_workbook
from pathlib import Path
import re
import numpy as np

<h3>Setting Regular Expressions</h3>

In [22]:
#Month Map
month_map = {
    'jan':1,'january':1,
    'feb':2,'february':2,
    'mar':3,'march':3,
    'apr':4,'april':4,
    'may':5,
    'jun':6,'june':6,
    'jul':7,'july':7,
    'aug':8,'august':8,
    'sep':9,'sept':9,'september':9,
    'oct':10,'october':10,
    'nov':11,'november':11,
    'dec':12,'december':12
}


# Reg expression patterns
# Year standalone
year_pattern = re.compile(r'(20\d{2})')

"""
# Full date (DD-MM-YYYY or DD/MM/YYYY)
full_date_pattern = re.compile(r'(\d{2})[-/](\d{2})[-/](20\d{2})')

# YYYY-MM or YYYY.MM or YYYY_MM
yyyy_mm_pattern = re.compile(r'(20\d{2})[._-](0[1-9]|1[0-2])')

# MM-YYYY or MM_YYYY
mm_yyyy_pattern = re.compile(r'(0[1-9]|1[0-2])[._-](20\d{2})')

# FY24 type
fy_pattern = re.compile(r'fy(\d{2})', re.IGNORECASE)

# Month Word
month_word_pattern = re.compile(
    r'(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec|'
    r'january|february|march|april|june|july|august|september|october|november|december)',
    re.IGNORECASE
)
"""

full_date_pattern = re.compile(r'(\d{1,2})[./-](\d{1,2})[./-](20\d{2})')
yyyy_mm_pattern = re.compile(r'(20\d{2})[._-](0[1-9]|1[0-2])')
mm_yyyy_pattern = re.compile(r'(0[1-9]|1[0-2])[._-](20\d{2})')
fy_pattern = re.compile(r'fy(\d{2})', re.IGNORECASE)

month_word_pattern = re.compile(
    r'(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec|'
    r'january|february|march|april|june|july|august|september|october|november|december)',
    re.IGNORECASE
)

year_pattern = re.compile(r'(20\d{2})')

month_short_year_pattern = re.compile(
    r'(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec|'
    r'january|february|march|april|june|july|august|september|october|november|december)'
    r'[\s\-_]*(\d{2})',
    re.IGNORECASE
)

month_apostrophe_year_pattern = re.compile(
    r'(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec|'
    r'january|february|march|april|june|july|august|september|october|november|december)'
    r"\s*[\'’]\s*(\d{2})",
    re.IGNORECASE
)
dd_mm_yyyy_loose_pattern = re.compile(r'(\d{1,2})[.](\d{1,2})[.](20\d{2})')

<>:22: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\d'
C:\Users\mailt\AppData\Local\Temp\ipykernel_18768\2931323100.py:22: SyntaxWarning: invalid escape sequence '\d'
  """


<h3>Month-Year Extraction</h3>

In [23]:
"""
def extract_year_month(filename):
    name = filename.lower()
    match = full_date_pattern.search(name)
    # FULL DATE
    if match:
        day, month, year = match.groups()
        return int(year), int(month)
        
    # YYYY-MM
    # -------------------------
    match = yyyy_mm_pattern.search(name)
    if match:
        year, month = match.groups()
        return int(year), int(month)
        
    # MM-YYYY
    # -------------------------
    match = mm_yyyy_pattern.search(name)
    if match:
        month, year = match.groups()
        return int(year), int(month)
        
    # FY + Month Word
    # -------------------------
    fy_match = fy_pattern.search(name)
    month_match = month_word_pattern.search(name)
    if fy_match and month_match:
        fy_year = int(fy_match.group(1))
        year = 2000 + fy_year
        month = month_map[month_match.group(0)]
        return year, month

    # Month Word + Year
    # -------------------------
    if month_match:
        month = month_map[month_match.group(0)]
        year_match = year_pattern.search(name)
        if year_match:
            year = int(year_match.group(0))
            return year, month

    return None, None
"""
def extract_year_month(filename):
    name = filename.lower()

    match = full_date_pattern.search(name)
    if match:
        day, month, year = match.groups()
        return int(year), int(month)

    # extra safety for dot format
    match = dd_mm_yyyy_loose_pattern.search(name)
    if match:
        day, month, year = match.groups()
        return int(year), int(month)

    match = yyyy_mm_pattern.search(name)
    if match:
        year, month = match.groups()
        return int(year), int(month)

    match = mm_yyyy_pattern.search(name)
    if match:
        month, year = match.groups()
        return int(year), int(month)

    fy_match = fy_pattern.search(name)
    month_match = month_word_pattern.search(name)

    if fy_match and month_match:
        fy_year = int(fy_match.group(1))
        year = 2000 + fy_year
        month = month_map[month_match.group(0)]
        return year, month

    if month_match:
        month = month_map[month_match.group(0)]
        year_match = year_pattern.search(name)
        if year_match:
            year = int(year_match.group(0))
            return year, month

    match = month_short_year_pattern.search(name)
    if match:
        month_word, short_year = match.groups()
        month = month_map[month_word]
        year = 2000 + int(short_year)
        return year, month

    match = month_apostrophe_year_pattern.search(name)
    if match:
        month_word, short_year = match.groups()
        month = month_map[month_word]
        year = 2000 + int(short_year)
        return year, month

    return None, None

<h3>Extracting Sheets</h3>

In [24]:
def get_sheets(file_path, extension):
    sheets = []
    try:
        if extension in [".xlsx", ".xls"]:
            wb = load_workbook(file_path, read_only= True)
            sheets = wb.sheetnames
        elif extension == ".xlsb":
            with open_workbook(file_path) as wb:
                sheets = wb.sheets
        elif extension == ".csv":
            sheets = ["CSV_SINGLE_SHEET"]
    except Extension as e:
        sheets = [f"ERROR: {str(e)}"]
    return sheets

<h3>Reading Files</h3>

In [25]:
month_name_map = {
    1:"JAN",2:"FEB",3:"MAR",4:"APR",5:"MAY",6:"JUN",
    7:"JUL",8:"AUG",9:"SEP",10:"OCT",11:"NOV",12:"DEC"
}

records = []
file_count = 0

ROOT_FOLDER = r"C:\Berry\RootFolder"

system_names = [
    name for name in os.listdir(ROOT_FOLDER)
    if os.path.isdir(os.path.join(ROOT_FOLDER, name))
]
for root, dirs, files in os.walk(ROOT_FOLDER):
    for file in files:
        file_count+=1
        if file.lower().endswith(('.csv', '.xlsx', '.xlsb', '.xls')):
            full_path = os.path.join(root, file)
            extension = os.path.splitext(file)[1].lower()
            parts = full_path.split(os.sep)
            try:
                system = parts[-3]
                year_folder = parts[-2]
            except:
                system = None
                year_folder = None
            extracted_year, extracted_month = extract_year_month(file)
            if extracted_month:
                month_str = f"{extracted_month:02d}"
                month_name = month_name_map[extracted_month]
            else:
                month_str = "NA"
                month_name = "NA"
            month_str = f"{extracted_month:02d}" if extracted_month else "NA"
            year_str = str(extracted_year) if extracted_year else "NA"
            sheet_list = get_sheets(full_path, extension)
            #print(f"{system:<12} : {file:<45}  -> {month_name} {year_str}")
            print(f"{system:<12} : {file:<45} [Sheet Count: {len(sheet_list)}] -> {month_name} {year_str}", sheet_list)
            records.append({
                "System": system,
                "FileName": file,
                "Month ": month_name,
                "Year": year_str,
                "Sheet Name": sheet_list
            })
df = pd.DataFrame(records)
df.to_csv("MetaData.csv", index=False)

System_A     : 2021_Jun_ALPHA_dump.xlsx                      [Sheet Count: 4] -> JUN 2021 ['README', 'Posting ', 'NOTES', 'Pivot']
System_A     : 2021_Oct_ALPHA_dump.xlsx                      [Sheet Count: 4] -> OCT 2021 ['README', 'Posting', 'NOTES', 'Pivot']
System_A     : ALPHA FY21 April Extract FINAL.xlsx           [Sheet Count: 4] -> APR 2021 ['README', 'Posting', 'NOTES', 'Pivot']
System_A     : ALPHA FY21 August Extract FINAL.xlsx          [Sheet Count: 4] -> AUG 2021 ['README', 'MAIN EXPORT ', 'NOTES', 'Pivot']
System_A     : ALPHA FY21 December Extract FINAL.xlsx        [Sheet Count: 4] -> DEC 2021 ['README', 'Main Export ', 'NOTES', 'Pivot']
System_A     : ALPHA-2021-03 raw.xlsx                        [Sheet Count: 4] -> MAR 2021 ['README', 'Posting', 'NOTES', 'Pivot']
System_A     : ALPHA-2021-09 raw.xlsx                        [Sheet Count: 4] -> SEP 2021 ['README', 'Transactions', 'NOTES', 'Pivot']
System_A     : ALPHA__Feb-2021__export (draft).xlsx          [Sheet Count:

<h3>Extracting Header Row</h3>

## CONFIG

In [27]:
ROOT_FOLDER = r"C:\BERRY\RootFolder"  
BRONZE_FOLDER = r"C:\BERRY\Bronze"   
MAPPING_FILE = r"sheet_mapping_by_file.csv"
OUTPUT_FILE = r"silver_sheet_inventory.csv"

## LOAD MAPPING

In [28]:
mapping_df = pd.read_csv(MAPPING_FILE)

mapping_dict = dict(
    zip(mapping_df['relative_path'], mapping_df['selected_sheet'])
)

## HEADER DETECTION FUNCTION (UNCHANGED)

In [30]:
def detect_header_position(df):

    best_score = -1
    best_index = None
    best_col = None

    for i in range(min(30, len(df))):

        row = df.iloc[i]

        if row.isnull().all():
            continue

        next_row = df.iloc[i+1] if i+1 < len(df) else None

        score = 0

        non_null_ratio = row.notnull().sum() / len(row)
        if non_null_ratio > 0.6:
            score += 2

        string_ratio = sum(isinstance(x, str) for x in row) / len(row)
        if string_ratio > 0.6:
            score += 2

        non_null_values = [str(x) for x in row if pd.notnull(x)]
        avg_length = sum(len(x) for x in non_null_values) / max(1, len(non_null_values))
        if avg_length < 20:
            score += 1

        keywords = ['id','name','date','amount','code','price','qty','total']
        keyword_found = any(
            any(k in str(cell).lower() for k in keywords)
            for cell in row if pd.notnull(cell)
        )
        if keyword_found:
            score += 2

        if len(set(non_null_values)) == len(non_null_values):
            score += 1

        if next_row is not None:
            numeric_count = sum(
                isinstance(x, (int, float)) for x in next_row if pd.notnull(x)
            )
            if numeric_count > len(next_row) * 0.3:
                score += 3

        if score > best_score:
            best_score = score
            best_index = i

            for j, val in enumerate(row):
                if pd.notna(val):
                    best_col = j
                    break

    if best_index is not None:
        #return best_index + 1, (best_col + 1 if best_col is not None else None)
        return best_index + 1, (best_col + 1 if best_col is not None else None), best_score

    return None, None, 0


## MAIN PROCESS

In [31]:
records = []

for relative_path, selected_sheet in mapping_dict.items():

    full_path = os.path.join(ROOT_FOLDER, relative_path)

    if not os.path.exists(full_path):
        print(f"File not found: {relative_path}")
        continue

    extension = Path(full_path).suffix.lower()

    parts = relative_path.split(os.sep)

    try:
        system = parts[0]
        year_folder = parts[1]
        file_name = parts[2]
    except:
        system = None
        year_folder = None
        file_name = os.path.basename(full_path)

    extracted_year, extracted_month = extract_year_month(file_name)

    # --------------------------------------------------
    # READ RAW DATA (FOR BOTH HEADER DETECTION + BRONZE SAVE)
    # --------------------------------------------------

    df = None

    try:
        if extension in ['.xlsx', '.xls']:
            df = pd.read_excel(full_path, sheet_name=selected_sheet, header=None, engine='openpyxl')

        elif extension == '.xlsb':
            df = pd.read_excel(full_path, sheet_name=selected_sheet, header=None, engine='pyxlsb')

        elif extension == '.csv':
            df = pd.read_csv(full_path, header=None)

    except Exception as e:
        print(f"Error reading file: {relative_path} | {e}")

    # --------------------------------------------------
    # HEADER DETECTION
    # --------------------------------------------------

    if df is not None:
        #header_row, header_col = detect_header_position(df)
        header_row, header_col, header_confidence = detect_header_position(df)
    else:
        header_row, header_col, header_confidence = None, None, 0

    # --------------------------------------------------
    # SAVE TO BRONZE 
    # --------------------------------------------------

    if df is not None:
        try:
            bronze_path = os.path.join(
                BRONZE_FOLDER,
                os.path.splitext(relative_path)[0] + ".csv"
            )

            os.makedirs(os.path.dirname(bronze_path), exist_ok=True)

            
            if header_row is not None and header_col is not None:
                df_clean = df.iloc[header_row-1:, header_col-1:]
                df_clean = df_clean.reset_index(drop=True)

                # Set first row as header
                df_clean.columns = df_clean.iloc[0]
                df_clean = df_clean[1:]
                df_clean = df_clean.reset_index(drop=True)
            else:
                df_clean = df.copy()

            df_clean.to_csv(bronze_path, index=False, encoding='utf-8-sig')

        except Exception as e:
            print(f"Error saving Bronze: {relative_path} | {e}")

    # --------------------------------------------------
    # RECORD METADATA
    # --------------------------------------------------

    records.append({
        "System": system,
        "YearFolder": year_folder,
        "FileName": file_name,
        "RelativePath": relative_path,
        "SelectedSheet": selected_sheet,
        "ExtractedYear": extracted_year,
        "ExtractedMonth": extracted_month,
        "HeaderRow_1Indexed": header_row,
        "HeaderCol_1Indexed": header_col,
        "Extension": extension
    })

    print("{:<10} | {:<40} | Sheet: {:<20} | Header: ({},{})".format(
        system or "",
        file_name or "NA",
        selected_sheet or "NA",
        header_row if header_row is not None else "NA",
        header_col if header_col is not None else "NA"
    ))

           | ALPHA__Jan-2021__export (draft).xlsx     | Sheet: Main                 | Header: (6,6)
           | ALPHA__Feb-2021__export (draft).xlsx     | Sheet: Extract              | Header: (12,1)
           | ALPHA-2021-03 raw.xlsx                   | Sheet: Posting              | Header: (3,5)
           | ALPHA FY21 April Extract FINAL.xlsx      | Sheet: Posting              | Header: (2,1)
           | May_2021__ALPHA__HIST.xlsx               | Sheet: LedgerView           | Header: (3,1)
           | 2021_Jun_ALPHA_dump.xlsx                 | Sheet: Posting              | Header: (14,2)
           | July_2021__ALPHA__HIST.xlsx              | Sheet: MonthlyData          | Header: (7,5)
           | ALPHA FY21 August Extract FINAL.xlsx     | Sheet: MAIN EXPORT          | Header: (13,2)
           | ALPHA-2021-09 raw.xlsx                   | Sheet: Transactions         | Header: (15,4)
           | 2021_Oct_ALPHA_dump.xlsx                 | Sheet: Posting              | Header: (1

## CREATE OUTPUT

In [32]:
inventory_df = pd.DataFrame(records)

inventory_df.to_csv(OUTPUT_FILE, index=False)

print("\nBronze + Inventory Generated Successfully.")
print("Total Processed Files:", len(inventory_df))


Bronze + Inventory Generated Successfully.
Total Processed Files: 96
